In [6]:
!pip install --quiet sentence_transformers==2.2.2 # NEED V2.2.2 CUZ DEPENDENCY ISSUES WITH LANGCHAIN
!pip install --quiet langchain
!pip install --quiet InstructorEmbedding
!pip install --quiet huggingface-cli
!pip install --quiet faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.7/973.7 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.9/307.9 kB 34.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 48.5 MB/s eta 0:00:00


In [7]:
!pip uninstall --quiet torch -y
!pip install --quiet torch==2.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.4 MB/s eta 0:00:00


In [8]:
# CHECKING IF GPU IS USED: CHANGE RUNTIME TO T4 GPU IF NOT
import torch
print(torch.__version__)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("GPU Available:", device)

2.2.1+cu121
GPU Available: cuda:0


In [ ]:
# LOGIN TO HUGGINGFACE TO GET ACCESS TO MODEL
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [10]:
# DOUBLE CHECK HUGGINGFACE LOGIN: SHOULD RETURN "bdsproject"
!huggingface-cli whoami

bdsproject


In [ ]:
# NEED AUTOGPTQ TO WORK WITH QUANTIZED LANGUAGE MODEL

# UNCOMMENT IF auto_gptq-0.4.0+cu118-cp310-cp310-linux_x86_64.whl IS NOT ALREADY IN THE FOLDER - SHOULD COME UPLOADED ALREADY
#!wget --quiet https://github.com/PanQiWei/AutoGPTQ/releases/download/v0.4.0/auto_gptq-0.4.0+cu118-cp310-cp310-linux_x86_64.whl

# INSTALL AUTOGPTQ LIBRARY
!pip install --quiet auto_gptq-0.4.0+cu118-cp310-cp310-linux_x86_64.whl

/content/gdrive/MyDrive/BDS Project - Irvin, Chris, Jerry/source_code/chatbot
auto_gptq-0.4.0+cu118-cp310-cp310-linux_x86_64.whl  db/  final_chatbot.ipynb


In [17]:
# MAY NEED TO INSTALL LANGCHAIN COMMUNITY IF LANGCHAIN NOT UP TO DATE
!pip install --quiet langchain-community langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.0 MB/s eta 0:00:00


In [18]:
# IMPORTING HELPFUL UTILITIES
from auto_gptq import AutoGPTQForCausalLM
from transformers import AutoTokenizer, TextStreamer, pipeline
from langchain import HuggingFacePipeline, PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFDirectoryLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
# NEED THIS TO WORK WITH TEXT FILE DIRECTORY LOADER
!pip install --upgrade --quiet "unstructured[all-docs]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 kB 39.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 60.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 96.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 52.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.6/459.6 kB 52.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 33.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 40.

In [ ]:
# LOAD TEXTSPLITTER AND EMBEDDINGS MODEL
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
embeddings = HuggingFaceInstructEmbeddings(model_name="all-MiniLM-L6-v2")

# GENERATE EMBEDDINGS FOR TEXT
def train_knowledgeBase(docs):
  texts = text_splitter.split_documents(docs)
  db = FAISS.from_documents(texts, embeddings)
  db.save_local("source_code/chatbot/db")
  return db

# IF ALREADY TRAINED, LOAD EXISTING DATABASE
def load_knowledgeBase():
  DB_FAISS_PATH = "source_code/chatbot/db"
  db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
  return db

# LOAD PDFS AND TEXT
def load_text():
  # LOAD PDFS OF SEC REPORTS AND CONVERT TO TEXT
  loader = PyPDFDirectoryLoader("data/test_pdfs")
  docs_pdf = loader.load()

  # LOAD GLASSDOOR REVIEW TEXT FILES
  loader_txt = DirectoryLoader("data/test_txts")
  docs_txt = loader_txt.load()

  # COMBINE SEC REPORTS AND REVIEWS
  docs = docs_txt + docs_pdf

  # UNCOMMENT BELOW LINES TO USE THE FULL DATASET - WILL NOT LOAD IN TIME
  #loader_8k = PyPDFDirectoryLoader("data/SEC/8k_filings_pdf")
  #docs_8k = loader_8k.load()
  #loader_10k = PyPDFDirectoryLoader("data/SEC/10k_filings_pdf")
  #docs_10k = loader_10k.load()
  #loader_txt = DirectoryLoader("data/glassdoor/text")
  #docs_txt = loader_txt.load()
  #docs = docs_txt + docs_8k + docs_10k

  return docs

# TRAIN FAISS EMBEDDINGS DATABASE - UNCOMMENT IF YOU ADD NEW DATA
#docs = load_text()
#db = train_knowledgeBase(docs)

# LOAD PREEXISTING KNOWLEDGE BASE
db = load_knowledgeBase()

/content/gdrive/MyDrive/BDS Project - Irvin, Chris, Jerry
data/  readme.md  reports/  source_code/
load INSTRUCTOR_Transformer
max_seq_length  512


In [23]:
# LOAD VERSION OF LLAMA2 MODEL QUANTIZED WITH GPTQ ALGORITHM
model_name = "TheBloke/Llama-2-13B-chat-GPTQ"

# LOAD TOKENIZER PRETRAINED ON SAME CORPUS AS MODEL ABOVE
# CONVERTS PROMPT INTO AND RESPONSE OUT OF A FORM READABLE TO THE MODEL
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# LOAD MODEL FOR CAUSAL LANGUAGE MODELING - GENERATING TEXT BASED ON PROMPT
model = AutoGPTQForCausalLM.from_quantized(
    model_name,                                # USE MODEL ABOVE
    revision='gptq-4bit-128g-actorder_True',   # SPECIFIC VERSION OF MODEL TO LOAD
    model_basename='model',                    # BASE NAME OF MODEL IN REPO
    use_safetensors=True,                      # MODEL WEIGHTS SERIALIZED
    trust_remote_code=True,                    # ALLOW RUNNING CODE IN MODEL'S REPO
    inject_fused_attention=False,              # SET TO TRUE FOR PERFORMANCE BENEFITS
    device='cuda:0',                           # USE GPU WE INITIALIZE ABOVE
    quantize_config=None,                      # NO ADDITIONAL QUANTIZATION PARAMS
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

1. You disabled CUDA extensions compilation by setting BUILD_CUDA_EXT=0 when install auto_gptq from source.
2. You are using pytorch without CUDA support.
3. CUDA and nvcc are not installed in your device.


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

quantize_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/7.26G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:4371: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
The cos_cached attribute will be removed in 4.39. Bear in mind that its contents changed in v4.38. Use the forward method of RoPE from now on instead. It is not used in the `LlamaAttention` class
The sin_cached attribute will be removed in 4.39. Bear in mind that its contents changed in v4.38. Use the forward method of RoPE from now on instead. It is not used in the `LlamaAttention` class


In [29]:
# INITIALIZE TEXT STREAMER FOR HANDLING INPUTS AND GIVING SEQUENTIAL OUTPUT INSTEAD OF FULL RESPONSE AT ONCE
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# INITIALIZE PIPELINE
text_pipeline = pipeline(
    "text-generation",
    model=model,                # USE OUR QUANTIZED LLAMA MODEL
    tokenizer=tokenizer,        # USE TOKENIZER WHICH USES THE SAME MODEL
    streamer=streamer,          # USE STREAMER ABOVE WHICH USES TOKENIZER
    max_new_tokens=512,         # SET MAXIMUM NUMBER OF TOKENS IN RESPONSE
    temperature=0.25,           # RANDOMNESS PARAMETER
    do_sample = True,
    top_p=0.95,                 # SAMPLE FROM TOKENS IN 95% OF PROB MASS
    repetition_penalty=1.15     # PENALIZE REPEATED TOKENS
)

# USING HUGGINGFACE PIPELINE
llm = HuggingFacePipeline(pipeline=text_pipeline)

The model 'LlamaGPTQForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo

In [30]:
# INITIALIZE DEFAULT SYSTE PROMPT GIVING INSTRUCTIONS
DEFAULT_SYSTEM_PROMPT = """
You are a helpful, respectful and honest assistant. Always answer as helpfully
as possible, while being safe. Your answers should not include any harmful,
unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your
responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain
why instead of answering something not correct. If you don't know the answer to a
question, please don't share false information.
""".strip()

# WRITE GENERAL PROMPT TEMPLATE USING CONTEXT AND QUESTION FOR RESPONSE
def generate_prompt(prompt, system_prompt = DEFAULT_SYSTEM_PROMPT) -> str:
    return f"""
[INST] <<SYS>>
{system_prompt}
<</SYS>>

{prompt} [/INST]
""".strip()

SYSTEM_PROMPT = "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer."

template = generate_prompt(
    """
{context}

Question: {question}
""",
    system_prompt=SYSTEM_PROMPT,
)

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

In [33]:
# FINAL FUNCTION TO GET RESPONSES
def get_response(query):
    # RETRIEVE MOST SIMILAR DOCS TO FORM CONTEXT
    similarSearch = db.similarity_search(query, k=4)
    new_faiss_index = FAISS.from_documents(documents=similarSearch, embedding=embeddings)
    retriever = new_faiss_index.as_retriever()

    # SET UP RETRIEVALQA CHAIN
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": prompt}
    )
    return qa_chain(query)

In [35]:
results = get_response("Based on the 8-k reports, 10-k reports, and Glassdoor reviews that you have access to (in your knowledge base), give a detailed analysis of NVIDIA’s core business model, financial status, and your forecast on the company’s performance. Give a conclusion on whether I should invest, and if I invest, what risks are involved?")

text [INST] <<SYS>>
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
<</SYS>>


information may be limited or incomplete, and our statements should not be read to indicate that we have conducted an exhaustive inquiry into, or review of, all
potentially available relevant information. These statements are inherently uncertain and investors are cautioned not to unduly rely upon these statements.
© 2024 NVIDIA Corporation. All rights reserved.
3

our investor relations website, http://investor.nvidia.com.
38

products; design, manufacturing or software defects; changes in consumer preferences or demands; changes in industry standards and interfaces; unexpected loss of performance
of our products or technologies when integrated into systems; as well as other factors detailed from time to time in the most recent reports NVIDIA files with the Securities and

files with the Securi

In [ ]:
result = get_response("Based on the 8-k reports, 10-k reports, and glassdoor reviews which you have access to (in your knowledge base), give a detailed analysis of Nvidia’s core business model,  financial status,  and your forecast on the company’s performance. Give a conclusion on whether I should invest and if I invest what risks are involved.")

 [INST] <<SYS>>
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
<</SYS>>


our investor relations website, http://investor.nvidia.com.
38

information may be limited or incomplete, and our statements should not be read to indicate that we have conducted an exhaustive inquiry into, or review of, all
potentially available relevant information. These statements are inherently uncertain and investors are cautioned not to unduly rely upon these statements.
© 2024 NVIDIA Corporation. All rights reserved.
3

products; design, manufacturing or software defects; changes in consumer preferences or demands; changes in industry standards and interfaces; unexpected loss of performance
of our products or technologies when integrated into systems; as well as other factors detailed from time to time in the most recent reports NVIDIA files with the Securities and

files with the Securities

In [ ]:
result = get_response("Imagine you are a financial advisor, and you are talking to your client, a rich guy named Irvin. He wants to know whether he should invest in NVIDIA or AMD. Based on your knowledge, give him your advice. Walk him through your analysis and tell him what risks are involved if he invests in either company. Remember you are talking as if you are a financial advisor and Irvin is your client. Explain things in simple terms, consider using analogies.")

In [ ]:
results = get_response("Conduct sentiment analysis based on Glassdoor Review of NVIDIA. Explain how the comments might affect the company's investment potential.")